# skyhunt — uruchomienie na Colab (GPU)

1. Wgraj nagrania do `MyDrive/skyhunt/raw/` (np. `DSCF4641.MOV`, `dark_frames.MOV`).
2. Uruchom komórki od góry.
3. Po rozłączeniu Colaba uruchom wszystko jeszcze raz: gotowe etapy są pomijane (`manifest.json` per plik na Drive).

Własny config: skopiuj `config.yaml` do `MyDrive/skyhunt/config.yaml`; ma pierwszeństwo przed wersją z repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = 'https://github.com/LukaszCiesielski-lodz/skytrack_uap.git'
BRANCH = 'main'
REPO = '/content/skytrack_uap'
DRIVE_ROOT = '/content/drive/MyDrive/skyhunt'
RAW = f'{DRIVE_ROOT}/raw'
OUT = f'{DRIVE_ROOT}/out'
CONFIG = f'{DRIVE_ROOT}/config.yaml' if os.path.exists(f'{DRIVE_ROOT}/config.yaml') else f'{REPO}/config.yaml'
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('config:', CONFIG)
print('nagrania:', sorted(os.listdir(RAW)))

In [ ]:
# Snapshot elementów orbit z CelesTrak (CSV/OMM) — uruchom jak najszybciej po nagraniu.
# CelesTrak udostępnia tylko BIEŻĄCE elementy, więc zamrażamy je na Drive (cache/gp/).
# Komórka nie zależy od pakietu skyhunt. Grupy pobrane < 2 h temu są pomijane;
# odpowiedź inna niż 200 przerywa pobieranie (CelesTrak blokuje IP po serii błędów).
import glob, time, urllib.request, urllib.error
from datetime import datetime, timezone

GP_DIR = f'{DRIVE_ROOT}/cache/gp'
GP_GROUPS = ['active', 'visual', 'stations', 'analyst',
             'cosmos-2251-debris', 'iridium-33-debris', 'fengyun-1c-debris']
os.makedirs(GP_DIR, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%MZ')
for g in GP_GROUPS:
    recent = [p for p in glob.glob(f'{GP_DIR}/celestrak_{g}_*.csv') if time.time() - os.path.getmtime(p) < 2 * 3600]
    if recent:
        print(f'{g:22s} pominięte (świeży plik: {os.path.basename(recent[0])})')
        continue
    url = f'https://celestrak.org/NORAD/elements/gp.php?GROUP={g}&FORMAT=csv'
    try:
        with urllib.request.urlopen(url, timeout=120) as r:
            data = r.read()
    except urllib.error.HTTPError as e:
        print(f'{g:22s} HTTP {e.code} — przerywam, spróbuj za 2 h')
        break
    path = f'{GP_DIR}/celestrak_{g}_{stamp}.csv'
    with open(path, 'wb') as fh:
        fh.write(data)
    print(f'{g:22s} {len(data):>10,d} B  {data.count(bytes([10])) - 1:>6d} obiektów')
    time.sleep(1)
print(sorted(os.listdir(GP_DIR)))

In [ ]:
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} fetch -q origin && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
!git -C {REPO} log -1 --oneline

In [ ]:
!pip install -q -e "{REPO}[gpu,astro,report,dev]"
!pip install -q PyNVVideoCodec || echo 'PyNVVideoCodec: instalacja nieudana (bench pokaże błąd)'
# torchcodec musi pasować do zainstalowanego torch: --no-deps, żeby nie podmienił torch Colaba.
import torch
CU = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
!pip install -q --no-deps torchcodec --index-url https://download.pytorch.org/whl/{CU} || echo 'torchcodec: niedostępny'
# astrometry.net (solve-field) do plate solve; indeksy 4110–4119 (~46 MB) pobiera etap `astrometry` do cache na Drive
!apt-get -qq update > /dev/null && apt-get -qq install -y astrometry.net > /dev/null && which solve-field
print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'GPU', torch.cuda.is_available())

In [ ]:
# Opcjonalnie: Space-Track (historia elementów i pełny katalog, w tym człony rakiet i śmieci).
# Dodaj w Colab → Secrets (ikona klucza) SPACETRACK_USER i SPACETRACK_PASSWORD; kod ich nie wypisuje.
try:
    from google.colab import userdata
    for key in ('SPACETRACK_USER', 'SPACETRACK_PASSWORD'):
        os.environ[key] = userdata.get(key)
    print('Space-Track: dane logowania ustawione')
except Exception:
    print('Space-Track: brak danych w Secrets — tylko CelesTrak')

## Testy
Testy `video` używają prawdziwego nagrania (parytet backendów bit w bit na 48 klatkach).

In [ ]:
TEST_VIDEO = f'{RAW}/DSCF4641.MOV'
os.environ['SKYHUNT_TEST_VIDEO'] = TEST_VIDEO if os.path.exists(TEST_VIDEO) else ''
!cd {REPO} && python -m pytest -q -rs

## Metadane i przepustowość dekodowania
Wynik benchmarku (`decode_bench.md`) wklej do README albo prześlij w czacie.

In [ ]:
!skyhunt probe "{RAW}" --config "{CONFIG}"

In [ ]:
## Pipeline
Etapy: `probe` → `stack` → `detect` → `tracks` → `astrometry` → `tle` → `identify` → `report`
(dla `dark_frames.MOV`: `darkstats` zamiast astrometrii i raportu). Po rozłączeniu uruchom ponownie — gotowe etapy są pomijane.
Raport: `MyDrive/skyhunt/out/<plik>/report/` — `summary.pdf`, `objects/sat_<NORAD>_t<id>.pdf`, `objects/unid_t<id>.pdf`, `clips/t<id>.mp4`.

## Pipeline
M0: `probe` (metadane, wstępny czas) i `stack` (średnia, max, statystyki per klatka).

In [ ]:
import glob, json
from IPython.display import Image, display

!skyhunt status "{RAW}" --config "{CONFIG}" --out "{OUT}"
for d in sorted(glob.glob(f'{OUT}/*/time_sync.json')):
    s = json.load(open(d))
    print(f"{d.split('/')[-2]}: Δ = {s['delta_s']:+.2f} ± {s['sigma_s']:.2f} s ({s['confidence']}, {s['method']})")
for p in sorted(glob.glob(f'{OUT}/*/report/**/*.pdf', recursive=True)):
    print(p)
for p in sorted(glob.glob(f'{OUT}/*/stack_max_minus_mean.png')):
    display(Image(p, width=1200))

In [ ]:
import glob
from IPython.display import Image, display

!skyhunt status "{RAW}" --config "{CONFIG}" --out "{OUT}"
for p in sorted(glob.glob(f'{OUT}/*/stack_max_minus_mean.png')):
    print(p)
    display(Image(p, width=1200))